# Volume of Mixing vs Pressure

Computes ΔV_mix(P*) = V_mixed − V_pure_solvent − V_pure_polymer
for 10 pressures from P*=0.8 to 2.0.

Reads volume data produced by the pressure_sweep.sh pipeline:
- **Mixed**:  from  (lx·ly·lz)
- **Pure solvent**:  (block-averaged volumes)
- **Pure polymer**:  (block-averaged volumes)

Manifest files in  point to each run directory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import glob
import os
from pathlib import Path

mpl.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 14,
    "figure.dpi": 120,
})

# --- Configuration ---
BASE_DATANAME  = "slab_support_5beads_tall_rho04"
INTERACTION    = "1.0_1.0"
PURE_INTER     = "1.0_0.0"
SLAB_STEPS     = 600000
PURE_STEPS     = 100000

# Local data root — one subfolder per pressure, e.g. flow_data_local/volmix_sweep/p1.0/
# Sync from Expanse: lammps_runs/volmix_sweep/{slab,sol,pol}_*/output_files/volume_data/*.dat
#                    and vol_pure_*.dat files into the matching p{P}/ folder.
DATA_DIR = Path("../../flow_data_local/volmix_sweep")

PRESSURES = [round(1.0 + i * 0.1, 1) for i in range(11)]
print(f"Pressures: {PRESSURES}")
print(f"Data root: {DATA_DIR.resolve()}")


In [ ]:
# === Sync volume data from Expanse ===
# Stages files on Expanse then rsyncs to flow_data_local/volmix_sweep/.
# Safe to re-run: rsync --update skips files already present and up to date.
import subprocess, sys

EXPANSE   = "dpollard@login.expanse.sdsc.edu"
STAGE_DIR = "~/Documents/lammps_runs/volmix_sweep/volmix_stage"

stage_script = r"""
STAGE=~/Documents/lammps_runs/volmix_sweep/volmix_stage
mkdir -p "$STAGE"
for P in 1.0 1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.0; do
  mkdir -p "$STAGE/p${P}"
  SLAB=$(ls -dt ~/Documents/lammps_runs/volmix_sweep/slab_with_support_*pstar${P}_* 2>/dev/null | head -1)
  SOL=$(ls  -dt ~/Documents/lammps_runs/volmix_sweep/solvent_pure_*pstar${P}_*       2>/dev/null | head -1)
  POL=$(ls  -dt ~/Documents/lammps_runs/volmix_sweep/polymer_pure_*pstar${P}_*       2>/dev/null | head -1)
  [ -n "$SLAB" ] && cp ${SLAB}/output_files/volume_data/box_dimensions_*.dat "$STAGE/p${P}/" 2>/dev/null || true
  [ -n "$SOL"  ] && cp ${SOL}/output_files/volume_data/box_dimensions_*.dat  "$STAGE/p${P}/" 2>/dev/null || true
  [ -n "$POL"  ] && cp ${POL}/output_files/volume_data/box_dimensions_*.dat  "$STAGE/p${P}/" 2>/dev/null || true
  echo "  P=${P}: $(ls \"$STAGE/p${P}/\" | wc -l) files staged"
done
"""

print("Step 1 — staging files on Expanse (you may be prompted for your password)...")
r1 = subprocess.run(["ssh", EXPANSE, "bash", "-s"], input=stage_script, text=True, capture_output=True)
print(r1.stdout)
if r1.returncode != 0:
    print("SSH error:\n", r1.stderr, file=sys.stderr)
else:
    print("Step 2 — rsyncing to local (skips files already up to date)...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    r2 = subprocess.run(
        ["rsync", "-av", "--update", f"{EXPANSE}:{STAGE_DIR}/", str(DATA_DIR) + "/"]
    )
    print("Sync complete." if r2.returncode == 0 else "rsync failed.", file=sys.stderr if r2.returncode != 0 else sys.stdout)


## Volume parsing functions

In [ ]:
def avg_box_volume(path, skip_frac=0.5):
    """
    Parse box_dimensions_*.dat (columns: step lx ly lz).
    Returns time-averaged volume = mean(lx*ly*lz) over the last (1-skip_frac) fraction.
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    step, lx, ly, lz = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                    data.append(lx * ly * lz)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def avg_pure_volume(path, skip_frac=0.0):
    """
    Parse vol_pure_*.dat (columns: step press_mean vol_mean rho_mean, block averages).
    Returns mean of vol_mean column over all blocks (skip_frac=0 since runs are short).
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    vol = float(parts[2])  # vol_mean column
                    data.append(vol)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


## Load volume data for each pressure

Uses the sweep manifest files written by pressure_sweep.sh to locate each run directory.

In [ ]:
results = []
missing = []

for P in PRESSURES:
    pstr = f"{P:.1f}"
    dataname     = f"{BASE_DATANAME}_pstar{pstr}"
    sol_dataname = f"final_config_{dataname}_{INTERACTION}_{SLAB_STEPS}_solvent_only"
    pol_dataname = f"final_config_{dataname}_{INTERACTION}_{SLAB_STEPS}_polymer_only"

    p_dir = DATA_DIR / f"p{pstr}"

    # Mixed system: box_dimensions from slab run (last 50% of run)
    slab_vol_file = p_dir / f"box_dimensions_{dataname}_{INTERACTION}_{SLAB_STEPS}.dat"

    # Pure solvent volume file
    sol_vol_file  = p_dir / f"box_dimensions_{sol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    # Pure polymer volume file
    pol_vol_file  = p_dir / f"box_dimensions_{pol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    if not all(f.exists() for f in [slab_vol_file, sol_vol_file, pol_vol_file]):
        missing.append(pstr)
        for label, fpath in [("slab", slab_vol_file), ("solvent", sol_vol_file), ("polymer", pol_vol_file)]:
            if not fpath.exists():
                print(f"[SKIP] P*={pstr}: missing {label} → {fpath}")
        continue

    try:
        V_mix = avg_box_volume(slab_vol_file, skip_frac=0.5)
        V_sol = avg_box_volume(sol_vol_file,  skip_frac=0.5)
        V_pol = avg_box_volume(pol_vol_file,  skip_frac=0.5)
        dV    = V_mix - V_sol - V_pol
        results.append({
            "P": P, "V_mix": V_mix, "V_sol": V_sol,
            "V_pol": V_pol, "dV_mix": dV
        })
        print(f"P*={pstr}:  V_mix={V_mix:.2f}  V_sol={V_sol:.2f}  V_pol={V_pol:.2f}  ΔV={dV:+.3f}")
    except Exception as e:
        print(f"[ERROR] P*={pstr}: {e}")
        missing.append(pstr)

df = pd.DataFrame(results)
print(f"\nLoaded {len(df)}/{len(PRESSURES)} pressure points")
if missing:
    print(f"Missing: {missing}")


## Plot ΔV_mix vs P*

In [ ]:
if df.empty:
    print("No data to plot — run pressure_sweep.sh and wait for all jobs to complete.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # --- Left: ΔV_mix vs P* ---
    ax = axes[0]
    ax.plot(df["P"], df["dV_mix"], "o-", color="steelblue", lw=2, ms=7)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"^*$")
    ax.set_ylabel(r"$\Delta V_{
m mix}\;[\sigma^3]$")
    ax.set_title("Volume of Mixing")
    ax.grid(True, alpha=0.3)

    # --- Right: individual volumes vs P* ---
    ax2 = axes[1]
    ax2.plot(df["P"], df["V_mix"], "o-", label=r"{
m mix}$", color="steelblue", lw=2, ms=6)
    ax2.plot(df["P"], df["V_sol"], "s--", label=r"{
m solvent}$", color="tomato",   lw=1.5, ms=5)
    ax2.plot(df["P"], df["V_pol"], "^--", label=r"{
m polymer}$", color="seagreen",  lw=1.5, ms=5)
    ax2.set_xlabel(r"^*$")
    ax2.set_ylabel(r"\;[\sigma^3]$")
    ax2.set_title("Component Volumes")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("volume_of_mixing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: volume_of_mixing.png")


## Summary table

In [ ]:
if not df.empty:
    display_df = df.copy()
    display_df.columns = ["P*", "V_mix [σ³]", "V_solvent [σ³]", "V_polymer [σ³]", "ΔV_mix [σ³]"]
    display_df = display_df.round(3)
    try:
        from IPython.display import display
        display(display_df.to_string(index=False))
    except ImportError:
        print(display_df.to_string(index=False))
